<a href="https://colab.research.google.com/github/ankitdsmb/ToolNexus/blob/main/ToolFactory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# ToolNexus AI Runtime (Colab)
# DeepSeek-Coder-6.7B + 4bit + Cloudflare + Auto Registration
# ===============================

# ===============================
# INSTALL DEPENDENCIES
# ===============================

!pip -q install fastapi uvicorn nest_asyncio transformers accelerate torch pydantic bitsandbytes requests

# ===============================
# INSTALL CLOUDFLARE
# ===============================

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 cloudflared

# ===============================
# IMPORTS
# ===============================

import nest_asyncio
import threading
import subprocess
import json
import torch
import re
import logging
import uuid
import time
import requests

from datetime import datetime
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import uvicorn

nest_asyncio.apply()

# ===============================
# CONFIGURATION
# ===============================

PORT = 8000
MAX_PROMPT_LENGTH = 512

RUNTIME_TOKEN = "toolnexus_secret_token"

MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"

# IMPORTANT: change to your backend API
TOOLNEXUS_API = "https://yourdomain.com/api/admin/ai/runtime/register"

HEARTBEAT_API = "https://yourdomain.com/api/admin/ai/runtime/heartbeat"

SYSTEM_TOKEN = "your_admin_secret"

# ===============================
# LOGGING
# ===============================

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ToolNexus-AI")

# ===============================
# RUNTIME METADATA
# ===============================

API_NAME = "ToolNexus-AI-Runtime"
API_VERSION = "1.0"
RUNTIME_ID = str(uuid.uuid4())[:8]

device = "cuda" if torch.cuda.is_available() else "cpu"

def api_stamp(endpoint, payload):
    return {
        "api": API_NAME,
        "version": API_VERSION,
        "runtimeId": RUNTIME_ID,
        "endpoint": endpoint,
        "timestamp": datetime.utcnow().isoformat(),
        "data": payload
    }

# ===============================
# MODEL LOADING
# ===============================

logger.info("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

logger.info("Model loaded successfully")

# ===============================
# MASTER PROMPT
# ===============================

MASTER_PROMPT = """
You are ToolNexus AI. Generate production-ready web tools.

Return ONLY valid JSON.

Schema:

{
  "toolName": "string",
  "slug": "string",
  "category": "string",
  "description": "string",
  "html": "string",
  "js": "string",
  "seoTitle": "string",
  "seoDescription": "string"
}
"""

# ===============================
# AI GENERATION
# ===============================

def run_ai(user_prompt):

    prompt = MASTER_PROMPT + "\nUser Request:\n" + user_prompt

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=1200,
            temperature=0.4,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    try:

        start = text.find('{')
        end = text.rfind('}')

        json_text = text[start:end+1]

        json_text = re.sub(r',\s*}', '}', json_text)

        return json.loads(json_text)

    except:

        return {
            "error": "invalid json",
            "raw": text[:400]
        }

# ===============================
# REQUEST MODEL
# ===============================

class GenerateRequest(BaseModel):

    token: str
    prompt: str = Field(..., max_length=MAX_PROMPT_LENGTH)
    requestId: str | None = None

# ===============================
# FASTAPI
# ===============================

app = FastAPI(title="ToolNexus AI Runtime")

@app.get("/ping")
async def ping():

    return api_stamp("/ping", {
        "status": "ok",
        "device": device
    })

@app.get("/health")
async def health():

    mem = torch.cuda.memory_allocated() if device == "cuda" else 0

    return api_stamp("/health", {
        "status": "healthy",
        "model": MODEL_NAME,
        "device": device,
        "memory": mem
    })

@app.post("/generate")
async def generate(req: GenerateRequest):

    if req.token != RUNTIME_TOKEN:

        raise HTTPException(status_code=401)

    request_id = req.requestId or str(uuid.uuid4())

    result = run_ai(req.prompt)

    return api_stamp("/generate", {
        "requestId": request_id,
        "success": True,
        "result": result
    })

# ===============================
# START SERVER
# ===============================

def run_server():

    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()

logger.info("API server started")

# ===============================
# START TUNNEL
# ===============================

logger.info("Starting tunnel")

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

public_url = None

for line in tunnel.stdout:

    match = re.search(r'(https://[a-zA-Z0-9\-\.]+\.trycloudflare\.com)', line)

    if match:

        public_url = match.group(1)
        break

# ===============================
# REGISTER RUNTIME
# ===============================

def register_runtime():

    payload = {
        "runtimeId": RUNTIME_ID,
        "endpoint": public_url,
        "model": MODEL_NAME,
        "device": device
    }

    headers = {
        "Authorization": f"Bearer {SYSTEM_TOKEN}"
    }

    try:

        r = requests.post(
            TOOLNEXUS_API,
            json=payload,
            headers=headers
        )

        logger.info("Runtime registered")

    except Exception as e:

        logger.warning("Registration failed")

if public_url:

    register_runtime()

# ===============================
# HEARTBEAT
# ===============================

def heartbeat():

    while True:

        try:

            requests.post(
                HEARTBEAT_API,
                json={"runtimeId": RUNTIME_ID},
                headers={"Authorization": f"Bearer {SYSTEM_TOKEN}"}
            )

        except:
            pass

        time.sleep(60)

threading.Thread(target=heartbeat, daemon=True).start()

# ===============================
# READY MESSAGE
# ===============================

print("\n==============================")
print("TOOLNEXUS AI RUNTIME READY")
print("==============================")
print("Local API:", f"http://localhost:{PORT}")

if public_url:
    print("Public URL:", public_url)

print("\nEndpoints")
print("/ping")
print("/health")
print("/generate")
print("==============================")